# Generate User Profile Summaries with Mistral-7B
**AdRec-GenAI / IEEE 2026 Paper**

Generates summaries for **1,411 evaluation users** (small_matrix) only — not all 7,176.
This gives you paper results in ~20 min instead of 3 hours.

```
big_matrix.csv   → 7,176 users  (training)
small_matrix.csv → 1,411 users  (evaluation)  ← we only need these
```

**Before running:**
1. Runtime → Change runtime type → **A100 GPU**
2. Upload to Google Drive at `My Drive/AdRec-GenAI/data/`:
   - `big_matrix.csv` (1.0 GB) — needed for watch history
   - `small_matrix.csv` (small) — defines which 1,411 users to process
   - `kuairec_caption_category.csv` (25 MB) — item text
3. Run all cells top to bottom


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
print('Done')

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/AdRec-GenAI/data'
OUT_DIR  = '/content/drive/MyDrive/AdRec-GenAI/embeddings'
os.makedirs(OUT_DIR, exist_ok=True)

for f in ['big_matrix.csv', 'small_matrix.csv', 'kuairec_caption_category.csv']:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) // (1024*1024) if exists else 0
    print(f'  {f}: {"✅ " + str(size) + " MB" if exists else "❌ NOT FOUND"}')

In [ ]:
# ── Cell 3: Load Mistral-7B with 4-bit quantization ───────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Loading {MODEL_ID} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto'
)
model.eval()
print('Model loaded ✅')

In [ ]:
# ── Cell 4: Build raw profiles for evaluation users only ──────────────────────
import pandas as pd
from tqdm.notebook import tqdm

TOP_K       = 20
ITEM_SEP    = '; '
DEDUPLICATE = True
FALLBACK    = 'no description'
FIELD_SEP   = ' | '
FIELDS = [
    ('manual_cover_text',          'cover'),
    ('caption',                    'caption'),
    ('topic_tag',                  'topics'),
    ('first_level_category_name',  'category1'),
    ('second_level_category_name', 'category2'),
]

def build_text(row):
    parts = [f'{p}: {row[c].strip()}' for c, p in FIELDS
             if c in row and isinstance(row[c], str) and row[c].strip()]
    return FIELD_SEP.join(parts) if parts else FALLBACK

# ── Load item metadata ─────────────────────────────────────────────────────────
print('Loading item metadata...')
meta_df = pd.read_csv(os.path.join(DATA_DIR, 'kuairec_caption_category.csv'),
                      sep=None, engine='python', on_bad_lines='skip')
meta_df.columns = [c.strip() for c in meta_df.columns]
vid_col = next(c for c in ['video_id','item_id','id'] if c in meta_df.columns)
meta_df[vid_col] = pd.to_numeric(meta_df[vid_col], errors='coerce')
meta_df = meta_df.dropna(subset=[vid_col]).copy()
meta_df[vid_col] = meta_df[vid_col].astype(int)
id2text = {int(row[vid_col]): build_text(row) for _, row in meta_df.iterrows()}
print(f'  {len(id2text):,} items')

# ── Get the 1,411 evaluation user IDs from small_matrix ───────────────────────
print('Loading small_matrix (evaluation users)...')
small = pd.read_csv(os.path.join(DATA_DIR, 'small_matrix.csv'),
                    usecols=['user_id'], dtype={'user_id':'int32'})
eval_user_ids = set(small['user_id'].unique())
print(f'  {len(eval_user_ids):,} evaluation users')

# ── Load big_matrix but only for those users ───────────────────────────────────
print('Loading big_matrix (watch history for eval users only)...')
big = pd.read_csv(
    os.path.join(DATA_DIR, 'big_matrix.csv'),
    dtype={'user_id':'int32', 'video_id':'int32'},
    usecols=['user_id', 'video_id', 'watch_ratio'],
)
big = big[big['user_id'].isin(eval_user_ids)]
print(f'  {len(big):,} interactions for {big["user_id"].nunique():,} users')

# ── Build profiles ─────────────────────────────────────────────────────────────
print('Building raw profiles...')
sorted_df = big.sort_values('watch_ratio', ascending=False)
user_ids, raw_texts = [], []
for uid, grp in tqdm(sorted_df.groupby('user_id'), desc='Profiles'):
    top_items = grp['video_id'].head(TOP_K).tolist()
    descs = [id2text.get(int(v), FALLBACK) for v in top_items]
    if DEDUPLICATE:
        seen, unique = set(), []
        for d in descs:
            if d not in seen: seen.add(d); unique.append(d)
        descs = unique
    user_ids.append(int(uid))
    raw_texts.append(ITEM_SEP.join(descs))
print(f'Built {len(user_ids):,} profiles ✅')

In [ ]:
# ── Cell 5: Generate summaries ─────────────────────────────────────────────────
import json

CACHE_PATH     = os.path.join(OUT_DIR, 'user_generative_summaries.json')
MAX_NEW_TOKENS = 80
TEMPERATURE    = 0.2
SAVE_EVERY     = 50

PROMPT_TEMPLATE = (
    'You are analyzing a short-video platform user. '
    'Based on the following videos they watched most, write a 2-sentence '
    'summary of their interests in plain English. Be concise.\n'
    'Videos: {raw_profile}\nSummary:'
)

# Resume from cache if exists
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f'Resuming: {len(cache["summaries"]):,} users already done')
else:
    cache = {'model': MODEL_ID, 'backend': 'colab_transformers',
             'n_users': len(user_ids),
             'generated_at': pd.Timestamp.utcnow().isoformat(),
             'summaries': {}}

summaries = cache['summaries']
remaining = [(uid, txt) for uid, txt in zip(user_ids, raw_texts)
             if str(uid) not in summaries]
print(f'{len(remaining):,} users remaining  |  ~{len(remaining)//60} min on A100')

def generate_summary(raw_profile):
    messages = [{'role': 'user', 'content': PROMPT_TEMPLATE.format(raw_profile=raw_profile)}]
    enc = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(enc, max_new_tokens=MAX_NEW_TOKENS,
                             temperature=TEMPERATURE, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.shape[1]:], skip_special_tokens=True).strip()

errors = 0
for i, (uid, raw_text) in enumerate(tqdm(remaining, desc='Generating')):
    try:
        summaries[str(uid)] = generate_summary(raw_text)
    except Exception as e:
        errors += 1
        summaries[str(uid)] = raw_text[:200]
        print(f'  Error user {uid}: {e}')
    if (i + 1) % SAVE_EVERY == 0:
        with open(CACHE_PATH, 'w') as f:
            json.dump(cache, f, indent=2, ensure_ascii=False)

with open(CACHE_PATH, 'w') as f:
    json.dump(cache, f, indent=2, ensure_ascii=False)

print(f'\n✅ {len(summaries):,} summaries saved to {CACHE_PATH}')
if errors: print(f'   ({errors} errors)')

In [ ]:
# ── Cell 6: Verify ────────────────────────────────────────────────────────────
print(f'Summaries: {len(summaries):,}')
print(f'File size: {os.path.getsize(CACHE_PATH)/1024:.0f} KB\n')
for uid_str, summary in list(summaries.items())[:3]:
    print(f'User {uid_str}:')
    print(f'  {summary}\n')

## Done — next steps on your Mac

**1. Download** `user_generative_summaries.json` from:
`My Drive/AdRec-GenAI/embeddings/`

**2. Put it here:**
```
AdRec-GenAI/kuairec/embeddings/user_generative_summaries.json
```

**3. Run locally:**
```bash
cd /Users/tanushreenepal/Desktop/AdRec-GenAI
python LLM-rec/src/build_user_llm_embeddings.py --use_generative
python LLM-rec/src/run_all.py
python LLM-rec/src/compare_metrics.py
```